# Exercise 6 — Process arriving files

**Worked solution** · [All exercises](../index.html) · [Setup](../README.md)

**Core: about 10 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 4 extra minutes; choose it here if the topic interests you.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Close the previous exercise after **Save and finish**. Missing earlier work? Use an explicit [catch-up step](../RECOVERY.md).

In [1]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales, enrich_sales, category_totals = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales', 'enrich_sales', 'category_totals')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
INCOMING = RUN_ROOT / "incoming"
INCOMING.mkdir()
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 17:35:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="exercise-6"></a>
## Your task

**Which parts change when the input keeps arriving?**

Core budget: about 10 minutes.

Read the initially empty `INCOMING` directory as a stream with `raw.schema`. Apply your existing functions in the same order. Keep `products` as the static lookup.

Start a named memory sink for the changing aggregate. The supplied Complete mode publishes the whole current report. Paths, query name and trigger are supplied.

### Your code — change the reader, reuse your transformations

In [2]:
stream_raw = spark.readStream.schema(raw.schema).parquet(spark_path(INCOMING))
stream_cleaned = clean_sales(stream_raw)
stream_accepted = accepted_sales(stream_cleaned)
stream_enriched = enrich_sales(stream_accepted, products)
stream_report = category_totals(stream_enriched)

In [3]:
print("Batch:", report.isStreaming, "Stream:", stream_report.isStreaming)
assert stream_report.isStreaming, "Use the streaming reader in Exercise 6."

Batch: False Stream: True


### Your code — start the query

A streaming DataFrame describes the computation; `start()` returns a running query. Do not call `show()` directly on the streaming DataFrame. We inspect the bounded memory table after processing each arrival.

The memory sink is for this classroom demonstration, not durable output. The two-second trigger is a schedule, not a latency guarantee. Keep this writer configuration unchanged for Exercise 7.

In [4]:
TABLE_NAME = "sales_" + uuid4().hex[:10]
CHECKPOINT_PATH = spark_path(RUN_ROOT / "report-checkpoint")

In [5]:
writer = (
    stream_report.writeStream.format("memory")
    .queryName(TABLE_NAME)
    .outputMode("complete")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="2 seconds")
)
query = writer.start()

### Supplied — first arrival

Run this cell once. The helper publishes completed files and refuses duplicates. `processAllAvailable()` waits for the finite input in this demonstration; the query keeps running afterwards.

After arrival 01: two accepted sales, total 65.00.

In [6]:
assert query.isActive
publish_arrival(DATA_ROOT, INCOMING, 1)
query.processAllAvailable()

In [7]:
spark.table(TABLE_NAME).orderBy("category").show()
check.arrival(spark.table(TABLE_NAME), 1)
print("Query active:", query.isActive)

+--------+-----+-----+
|category|sales|total|
+--------+-----+-----+
|   books|    1|25.00|
|   games|    1|40.00|
+--------+-----+-----+

Arrival 1 verified: 2 sales; total 65.00.
Query active: True


### Predict, then publish the second arrival

Arrival 02 contains s3, s4 and s6. How many should enter the report? Predict its total before running.

Your prediction: …

In [8]:
publish_arrival(DATA_ROOT, INCOMING, 2)
query.processAllAvailable()

In [9]:
spark.table(TABLE_NAME).orderBy("category").show()
check.arrival(spark.table(TABLE_NAME), 2)

+--------+-----+-----+
|category|sales|total|
+--------+-----+-----+
|   books|    2|40.00|
|   games|    1|40.00|
|unmapped|    1|10.00|
+--------+-----+-----+

Arrival 2 verified: 4 sales; total 90.00.


<details>
<summary>Need a nudge? Hint 1</summary>

The reader changes, but cleaning, filtering, the static lookup join and aggregation still use your functions.

</details>

<details>
<summary>A little more help: Hint 2</summary>

Output modes describe emitted result rows. The memory table should contain every current category total after an update.

</details>

If you need to catch up during class, use the explicit [recovery step](../RECOVERY.md#exercise-6). [Worked solution](06-streaming.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 4 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### Three different objects

- `stream_report`: an unbounded DataFrame description.
- `writer`: output configuration, including the checkpoint and query name.
- `query`: the active execution, with `isActive`, `lastProgress` and `stop()`.

`spark.table(TABLE_NAME)` reads the current bounded debugging result. A file arrival is not a promise of one micro-batch; checks run after the available input has been processed. Keep the static lookup and transformation definitions unchanged while the query runs.

### Inspect a completed batch

Run the supplied inspection cell. Which object is a description, which configures output, and which is running? Find the batch ID and the number of input rows in `lastProgress`. The latest completed batch may contain no new rows; a file is not a promised batch boundary.

Your observations: …

In [10]:
print("Streaming DataFrame:", stream_report.isStreaming)
print("Query active:", query.isActive)
progress = query.lastProgress
print("Last completed batch:", progress["batchId"], "input rows:", progress["numInputRows"])
print("Source descriptions:", [item["description"] for item in progress["sources"]])

Streaming DataFrame: True
Query active: True
Last completed batch: 1 input rows: 3
Source descriptions: ['FileStreamSource[file:<lab-root>/runs/run-ec55513c27/incoming]']


<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [11]:
check.arrival(spark.table(TABLE_NAME), 2)
query.stop()
workspace.remember_stream(RUN_ROOT, TABLE_NAME, 2)
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Arrival 2 verified: 4 sales; total 90.00.


Session stopped; exercise files are under runs/run-ec55513c27


Next: [Exercise 7 — Resume from a checkpoint](07-checkpoint.ipynb).